## 《DETR3D: 3D Object Detection from Multi-view Images via 3D-to-2D Queries》

- 这篇论文提出了一种基于 transformer 的新型 3D 目标检测架构，核心思想是使用 **3D-to-2D 查询机制**，跳过传统深度估计或 point cloud 重建过程，直接在 **3D 空间中进行检测**，以提升精度和效率。


### 总体流程（Figure 1）

输入：多视角图像 + 相机内外参
输出：3D 边界框（位置、尺寸、朝向、速度、类别）

关键模块：

* **ResNet + FPN**：提取图像多尺度特征
* **Object Queries（稀疏3D位置）**：初始化一组“对象假设”，作为 transformer 的输入
* **3D 参考点 → 2D 图像空间投影**：通过相机矩阵将 3D 位置投影到多视图图像中，索引特征
* **特征聚合 + 多头注意力**：融合来自多个视角的信息，更新 object query
* **bounding box + 分类预测**：每一层都可预测，最终层输出用于推理
* **Set-to-set 匹配 loss（无NMS）**：与 DETR 类似，采用 Hungarian 匹配优化目标框


## 核心创新点分析

### 1. Top-down 查询机制（3D-to-2D）

以 3D 空间中固定的 object queries 为中心，投影到多个摄像头图像空间中，再采样图像特征，而不是从每个像素点估计 3D 坐标（bottom-up），**通过相机内参矩阵将3D参考点坐标转为2D，通过grid——sample获得对应点的值**。

- 将sample_feature与query进行融合，计算注意力。
**优势**：

* 跳过 dense depth map，避免 compounding error（深度错误 → 检测错误）
* 目标位置与图像特征通过几何关系直接关联，不再依赖显式深度估计模块

### 2. Query refinement 机制（多层迭代更新）

每个 layer：

1. object query → 3D 参考点（由子网络预测）
2. 将参考点投影到多视角图像（使用相机投影矩阵）
3. 从各视角图像特征图中采样点对应的图像特征（bilinear 插值）
4. 将特征加权融合进 query，送入 multi-head attention，生成 refined query

注意：参考点在不可见区域会被过滤（通过 σ 值掩码）

最终每个 query 输出 3D 边界框 + 类别标签预测，训练时在所有层上监督；推理只用最后一层。

## 一、整体关键模块结构复述

- 多视角图像输入
- 图像特征提取 Backbone (如 ResNet-FPN)
- 多尺度图像特征 {F_l}（通常为4个层级） 每个 F_l ∈ [B, C, H_l, W_l]
- 初始化 3D object queries  [num_queries, d_model] 
- Query refinement 模块（多层 Transformer）
- 分类 & 3D BBox 预测头

## 二、Query Refinement 模块（迭代式更新）

这个模块是 DETR3D 的核心亮点，它每一层都通过 **"query → reference point → 2D 采样 → 特征融合 → 新 query"** 的过程来优化预测。

---

### 输入格式说明

1. **图像特征 F\_l**（共 L 层）

   * FPN 多尺度输出，每层形状为：

     ```
     F_l: [B, C, H_l, W_l]
     ```
   * 通常有 4 层特征图（P2 \~ P5）

2. **Object Queries**

   * 初始为 learnable embedding：

     ```
     queries: [num_queries, d_model]
     ```
   * 常设：num\_queries=900，d\_model=256

3. **相机参数（标定）**

   * 提供每帧中各视角图像的相机内参/外参，进行 3D → 2D 投影用

---

### 每一层的处理流程及维度变换

每层 Transformer Layer 执行：

---

#### Step 1：生成当前参考点（Reference Points）

每个 query 预测一个归一化 3D 点（在鸟瞰图空间）：

```
reference_points: [B, num_queries, 3]  # (x, y, z) in BEV normalized coords
```

---

#### Step 2：将参考点投影到每个摄像头图像平面

对于每个相机、每个 query：

* 使用相机内外参将 reference point 从 3D 世界坐标 → 相机坐标 → 2D 像素点
* 得到采样位置：

  ```
  projected_uv: [B, num_queries, num_cameras, 2]  # 2D 图像坐标
  ```

- **！如果某参考点不在某个摄像头视野内，会设置为无效/掩码。**

---

#### Step 3：从多视角图像特征图中采样特征

* 使用 bilinear 插值，从每个摄像头每一层特征图中采样：

  ```
  sampled_feats: [B, num_queries, num_cameras, L, C]  # L = 特征层数
  ```
* 多尺度采样后，通常会通过一个线性层或加权平均融合，变成一个：

  ```
  aggregated_feats: [B, num_queries, d_model]
  ```

---

#### Step 4：将采样特征送入 Multi-Head Attention，与当前 query 融合

1. 使用 **cross attention**：

   ```
   q: current query         → [B, num_queries, d_model]
   k,v: sampled_feat tokens → [B, num_queries, d_model]
   ```

2. 输出更新后的 query：

   ```
   refined_query: [B, num_queries, d_model]
   ```

这一步就像把 query 聚焦在多视角图像空间的“投影点”，提取局部上下文信息更新自身。

---

#### Step 5：加上位置编码、送入 FFN

最终这一层的 query 更新后通过 FFN：

```
FFN(refined_query) → 输出分类分数 + bbox 参数
```

DETR3D 通常在每一层 decoder 都预测一次，最后一层用于推理。

---

## 总结维度追踪表

| 名称            | 尺寸说明                                   | 备注                       |
| ------------- | -------------------------------------- | ------------------------ |
| 图像特征 F\_l     | \[B, C, H\_l, W\_l]                    | 多层级                      |
| Object query  | \[B, num\_queries, d\_model]           | 初始化 learnable embedding  |
| 参考点 ref\_pts  | \[B, num\_queries, 3]                  | 在每层动态生成                  |
| 投影点 uv        | \[B, num\_queries, num\_cameras, 2]    | 由 ref\_pts + cam\_mat 得到 |
| 采样图像特征        | \[B, num\_queries, num\_cameras, L, C] | 多尺度图像融合信息                |
| 聚合后图像特征       | \[B, num\_queries, d\_model]           | 可用于 cross-attn           |
| refined query | \[B, num\_queries, d\_model]           | 用于下层循环                   |




## 与其他方法对比优势

| 方法               | 特点                                    | 弱点                                               |
| ---------------- | ------------------------------------- | ------------------------------------------------ |
| **FCOS3D**       | 2D anchor-free 网络 + depth 预测回归 3D box | 深度估计误差显著影响下游精度；需要 NMS                            |
| **pseudo-LiDAR** | 图像 → depth → 点云 → PointNet            | 误差累积；depth 网络需要额外训练                              |
| **DETR3D**       | 不预测 depth，直接从 sparse query 提取图像特征     | 对 reference point 投影机制敏感；对长距离物体仍有 translation 误差 |

在 nuScenes 的重叠区域中，DETR3D 优势尤为明显，因其一次性利用了全部相机图像信息。

---

## 关键实验设计（Ablation Insights）

1. **层数迭代**：越深层越接近真实 bbox（参见 Figure 2 和 Table 5）
2. **query 数量**：太少表现差；>900 无显著提升（Table 6）
3. **backbone 比较**：ResNet101 性能优于 ResNet50（Table 7）

---

## 总结要点

| 特性          | DETR3D 优势                  |
| ----------- | -------------------------- |
| Depth-Free  | 不再依赖 depth map 或 LiDAR 数据  |
| Top-down    | 以 3D query 主导整个检测过程        |
| 全视角融合       | 一次性利用所有相机图像（无 late fusion） |
| 无需后处理       | 不需要 NMS，输出即最终结果            |
| Transformer | Query 与图像特征间的高效匹配与交互       |

